In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
REPO = Path.home() / "Code" / "processing" / "riboseq"
PROJECT = Path("/mnt/d/Ibnu/riboseq/AT")

TABLES = PROJECT / "TABLES"

SALMON_IN = TABLES / "salmon_transcript.10pct.tsv"
TE_OUT = TABLES / "translation_efficiency.10pct.tsv"

In [3]:
salmon = pd.read_csv(SALMON_IN, sep="\t")

print(salmon.shape)
salmon.head()

(59051, 8)


,Transcript_ID,Gene_ID,Ribo_NumReads,RNA_NumReads,Ribo_TPM,RNA_TPM,Ribo_Length,RNA_Length
0,AT1G01010.1,AT1G01010,2.0,19.000,0.111252,3.472288,1438.0,1438.0
1,AT1G01020.2,AT1G01020,0.0,24.875,0.000000,7.810271,837.0,837.0
2,AT1G01020.6,AT1G01020,0.0,0.000,0.000000,0.000000,694.0,694.0
3,AT1G01020.1,AT1G01020,0.0,68.125,0.000000,16.592177,1079.0,1079.0
4,AT1G01020.3,AT1G01020,0.0,0.000,0.000000,0.000000,1170.0,1170.0


In [4]:
# ============================================
# Calculate residual translation efficiency
# ============================================

te = salmon.copy()

# Compositional abundances from raw Salmon estimated counts
te["RNA_prop"] = te["RNA_NumReads"] / te["RNA_NumReads"].sum()
te["Ribo_prop"] = te["Ribo_NumReads"] / te["Ribo_NumReads"].sum()

# Use only transcripts with non-zero RNA and Ribo signal
fit_mask = (
    (te["RNA_prop"] > 0) &
    (te["Ribo_prop"] > 0)
)

te["log2_RNA_prop"] = np.nan
te["log2_Ribo_prop"] = np.nan
te["Expected_log2_Ribo_prop"] = np.nan
te["TE_residual"] = np.nan

te.loc[fit_mask, "log2_RNA_prop"] = np.log2(te.loc[fit_mask, "RNA_prop"])
te.loc[fit_mask, "log2_Ribo_prop"] = np.log2(te.loc[fit_mask, "Ribo_prop"])

x = te.loc[fit_mask, "log2_RNA_prop"].to_numpy()
y = te.loc[fit_mask, "log2_Ribo_prop"].to_numpy()

slope, intercept = np.polyfit(x, y, 1)

te.loc[fit_mask, "Expected_log2_Ribo_prop"] = (
    intercept + slope * te.loc[fit_mask, "log2_RNA_prop"]
)

te.loc[fit_mask, "TE_residual"] = (
    te.loc[fit_mask, "log2_Ribo_prop"] -
    te.loc[fit_mask, "Expected_log2_Ribo_prop"]
)

print("Regression slope:", slope)
print("Regression intercept:", intercept)
print("Transcripts used:", fit_mask.sum())

te.head()

Regression slope: 0.621475889267534
Regression intercept: -6.734226004584542
Transcripts used: 14681


,Transcript_ID,Gene_ID,Ribo_NumReads,RNA_NumReads,Ribo_TPM,RNA_TPM,Ribo_Length,RNA_Length,RNA_prop,Ribo_prop,log2_RNA_prop,log2_Ribo_prop,Expected_log2_Ribo_prop,TE_residual
0,AT1G01010.1,AT1G01010,2.0,19.000,0.111252,3.472288,1438.0,1438.0,0.000004,0.000003,-17.78561,-18.490001,-17.787554,-0.702447
1,AT1G01020.2,AT1G01020,0.0,24.875,0.000000,7.810271,837.0,837.0,0.000006,0.000000,NaN,NaN,NaN,NaN
2,AT1G01020.6,AT1G01020,0.0,0.000,0.000000,0.000000,694.0,694.0,0.000000,0.000000,NaN,NaN,NaN,NaN
3,AT1G01020.1,AT1G01020,0.0,68.125,0.000000,16.592177,1079.0,1079.0,0.000016,0.000000,NaN,NaN,NaN,NaN
4,AT1G01020.3,AT1G01020,0.0,0.000,0.000000,0.000000,1170.0,1170.0,0.000000,0.000000,NaN,NaN,NaN,NaN


In [5]:
te.to_csv(TE_OUT, sep="\t", index=False)

print(TE_OUT)
print("Saved:", TE_OUT.exists())
print(te.shape)

/mnt/d/Ibnu/riboseq/AT/TABLES/translation_efficiency.10pct.tsv
Saved: True
(59051, 14)
